# Figure 1

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle, Wedge

from lib import N_OPTIMIZERS, OPTIMIZER_OVERVIEW_PATH, OUTPUT_DIR

optimizer_df = pd.read_csv(OPTIMIZER_OVERVIEW_PATH).query(
    "~is_pysacess and ~excluded"
)
assert len(optimizer_df) == N_OPTIMIZERS

In [ ]:
optimizer_df.head()

In [ ]:
# Generate features for donut chart
optimizer_df["scheme"] = optimizer_df.package.apply(
    lambda x: "eSS" if x in {"pyscat", "SaCeSS"} else "multi-start"
)


def set_scope(row):
    if "MLSL" in row["optimizer_label"]:
        return "hybrid"

    if row["optimizer_label"] in ("SaCeSS+Ipopt", "SaCeSS+DHC"):
        # classification is based on the innermost method
        return "local"

    return (
        "global" if row["is_global"] else "local" if row["is_local"] else "NA"
    )


optimizer_df["scope"] = optimizer_df.apply(set_scope, axis=1)

assert optimizer_df.hessian_based.notna().all()
assert optimizer_df.gradient_based.notna().all()


def set_deriv(row):
    if row["hessian_based"] or row["gradient_based"]:
        return "derivative-based"

    return "derivative-free"


optimizer_df["derivative"] = optimizer_df.apply(set_deriv, axis=1)
optimizer_df.set_index("optimizer_label")

In [ ]:
df_chart = optimizer_df[
    ["optimizer_label", "scheme", "scope", "derivative"]
].rename(
    columns={
        "scheme": "strategy",
        "optimizer_label": "optimizer",
    }
)
df_chart.set_index("optimizer", inplace=True)

# make categorical with fixed-order to ensure right sorting after groupby below
df_chart.strategy = pd.Categorical(
    df_chart.strategy, ordered=True, categories=["multi-start", "eSS"]
)
df_chart.scope = pd.Categorical(
    df_chart.scope, ordered=True, categories=["local", "hybrid", "global"]
)
df_chart.derivative = pd.Categorical(
    df_chart.derivative,
    ordered=True,
    categories=[
        "derivative-free",
        "derivative-based, 2nd order",
        "derivative-based, 1st order",
        "derivative-based",
    ],
)

In [ ]:
def points_to_data(ax, points):
    """Transform distance from pt to data."""
    # assumes equal aspect
    fig = ax.figure

    # points -> pixels
    pixels = points * fig.dpi / 72.0

    # axis height in pixels
    bbox = ax.get_window_extent()
    height_px = bbox.height

    # y-range in data coordinates
    y0, y1 = ax.get_ylim()
    height_data = abs(y1 - y0)

    return pixels * height_data / height_px


def draw_ring(
    ax: plt.Axes,
    groups,
    radius,
    width,
    fontsize=8,
    start_angle=90,
    colors=None,
    label=True,
    edgecolor="white",
):
    total = sum(v for _, v in groups)
    angle = start_angle

    spans = {}

    for name, value in groups:
        theta = 360 * value / total
        theta1, theta2 = angle, angle - theta

        color = colors.get(name, "lightgray") if colors else "lightgray"
        wedge = Wedge(
            (0, 0),
            radius,
            theta2,
            theta1,
            width=width,
            facecolor=color,
            edgecolor=edgecolor,
            linewidth=1,
        )
        ax.add_patch(wedge)

        mid = np.deg2rad((theta1 + theta2) / 2)
        spans[name] = (theta1, theta2)
        # if the segment includes 6 o'clock (-90°), put the label there
        if theta1 > -45 and theta2 < -135:
            mid = np.deg2rad(-90)

        if label and value > 0:
            r = radius - width / 2
            ax.text(
                r * np.cos(mid),
                r * np.sin(mid),
                f"{name}\n(n={value})",
                ha="center",
                va="center",
                fontsize=fontsize,
                # rotation=np.rad2deg(mid) - 90,
                rotation=np.rad2deg(mid) - 90 - 180
                if -90 <= np.rad2deg(mid) < 0
                else np.rad2deg(mid) - 90,
                rotation_mode="anchor",
            )
            if False:
                r = (
                    radius
                    - width / 2
                    + 1.5
                    * points_to_data(ax, fontsize)
                    * (-1 if -90 <= np.rad2deg(mid) < 0 else 1)
                )
                ax.text(
                    r * np.cos(mid),
                    r * np.sin(mid),
                    name,
                    ha="center",
                    va="center",
                    fontsize=fontsize,
                    # rotation=np.rad2deg(mid) - 90,
                    rotation=np.rad2deg(mid) - 90 - 180
                    if -90 <= np.rad2deg(mid) < 0
                    else np.rad2deg(mid) - 90,
                    rotation_mode="anchor",
                )
                r = (
                    radius
                    - width / 2
                    - 1.5
                    * points_to_data(ax, fontsize)
                    * (-1 if -90 <= np.rad2deg(mid) < 0 else 1)
                )
                ax.text(
                    r * np.cos(mid),
                    r * np.sin(mid),
                    f"(n={value})",
                    ha="center",
                    va="center",
                    fontsize=fontsize - 1,
                    # rotation=np.rad2deg(mid) - 90,
                    rotation=np.rad2deg(mid) - 90 - 180
                    if -90 <= np.rad2deg(mid) < 0
                    else np.rad2deg(mid) - 90,
                    rotation_mode="anchor",
                )
        angle = theta2

    return spans


def draw_leaf_labels(ax, labels, fontsize=8, radius=1.25, start_angle=90):
    n = len(labels)
    angle = start_angle
    theta = 360 / n

    for label in labels:
        mid = np.deg2rad(angle - theta / 2)
        x, y = radius * np.cos(mid), radius * np.sin(mid)
        rot = np.rad2deg(mid)

        ha = "left" if x >= 0 else "right"
        rotation = rot if x >= 0 else rot + 180

        ax.text(
            x,
            y,
            label,
            ha=ha,
            va="center",
            fontsize=fontsize,
            rotation=rotation,
            rotation_mode="anchor",
        )

        angle -= theta


def optimizer_donuts(
    df,
    radius_center=0.18,
    gap=0.05,
    donut_width=0.22,
    fontsize_label=10,
    fontsize_leaf=10,
    ax: plt.Axes = None,
):
    abbrevs = {
        "derivative-free": "DF",
        "derivative-based": "DB",
        "derivative-based, 2nd order": "DB2",
        "derivative-based, 1st order": "DB1",
    }
    abbrev_threshold = 2

    # sort: inner to outer donut
    col_order = ["strategy", "scope", "derivative"]
    df = df.sort_index().sort_values(col_order, ascending=[True, True, True])

    # Counts
    level1 = df.groupby(col_order[0]).size()
    level2 = df.groupby(col_order[:2]).size()
    level3 = df.groupby(col_order).size()

    for level in [level1, level2, level3]:
        # if count is <= abbrev_threshold: replace label by abbrev
        level.index = [
            tuple([*label[:-1], abbrevs.get(label[-1], label[-1])])
            if count <= abbrev_threshold and isinstance(label, tuple)
            else label
            for label, count in level.items()
        ]
    leaf_labels = df.index.tolist()

    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={"aspect": "equal"})
    ax.set_aspect("equal")
    ax.axis("off")

    # Center
    center = Circle((0, 0), radius_center, color="#888888")
    ax.add_patch(center)
    ax.text(
        0,
        0,
        f"optimisation\nmethods\n(n={len(df)})",
        ha="center",
        va="center",
        color="white",
        fontsize=fontsize_label,
    )

    # Inner ring
    draw_ring(
        ax,
        list(level1.items()),
        radius=radius_center + donut_width + gap,
        width=donut_width,
        colors={
            "multi-start": "#ffd34d",
            "eSS": "#ffbf3f",
        },
        fontsize=fontsize_label,
    )

    # Middle ring
    middle_groups = [(f"{b}", v) for (a, b), v in level2.items()]

    draw_ring(
        ax,
        middle_groups,
        radius=radius_center + 2 * (donut_width + gap),
        width=donut_width,
        colors={
            "local": "#8ee36f",
            "hybrid": "#61C84D",
            "global": "#5FA44B",
        },
        fontsize=fontsize_label,
    )

    # Outer category ring
    outer_groups = [(f"{c}", v) for (a, b, c), v in level3.items()]
    draw_ring(
        ax,
        outer_groups,
        radius=radius_center + 3 * (donut_width + gap),
        width=donut_width,
        colors={
            k: "#4db6e8" if "derivative-based" in k or "DB" in k else "#7ec8ee"
            for k, _ in outer_groups
        },
        fontsize=fontsize_label,
    )

    # Optimizer labels around outside
    draw_leaf_labels(
        ax,
        leaf_labels,
        radius=radius_center + 3 * (donut_width + gap) + gap,
        fontsize=fontsize_leaf,
    )

    ax.set_xlim(-1.55, 1.55)
    ax.set_ylim(-1.55, 1.55)

In [ ]:
def label_bars(bars, ax):
    """Label bars with their height."""
    max_bar_height = max(bar.get_height() for bar in bars)
    y_offset = max(0.1, max_bar_height * 0.01)
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + y_offset,
            f"n={int(height)}",
            ha="center",
            va="bottom",
            fontsize=mpl.rcParams["xtick.labelsize"],
            clip_on=False,
        )


def subfigs_bc(df, ax1, ax2):
    assert len(df) == N_OPTIMIZERS

    package_counts = df["package"].value_counts()
    objective_counts = df["objective_impl"].value_counts()

    bar_width = 0.8
    x_step = 1.0
    gap = x_step - bar_width

    # maximum number of bars across rows
    max_len = max(len(package_counts), len(objective_counts))

    # bar chart -- optimizers by package
    n1 = len(package_counts)
    x1 = gap + np.arange(n1) * x_step

    cmap = plt.get_cmap("tab10")
    colors = cmap(np.linspace(0, 1, len(package_counts)))

    bars1 = ax1.bar(
        x1,
        package_counts.values,
        width=bar_width,
        color=colors,
        edgecolor="black",
    )
    ax1.set_xlim(0 - bar_width / 2, max_len * x_step - (1 - bar_width) / 2)

    ax1.set_ylabel("# Optimisation methods")
    ax1.set_xticks(x1)
    ax1.set_xticklabels(package_counts.index, rotation=45, ha="right")

    label_bars(bars1, ax1)

    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)
    ax1.spines["left"].set_bounds(0, max(package_counts.values))

    # bar chart -- optimizers by objective implementation
    n2 = len(objective_counts)
    x2 = gap + np.arange(n2) * x_step
    colors2 = ["#D783FF", "#FF8AD8"]
    assert n2 == len(colors2)
    bars2 = ax2.bar(
        x2,
        objective_counts.values,
        color=colors2,
        width=bar_width,
        edgecolor="black",
    )
    ax2.set_xlim(0 - bar_width / 2, max_len * x_step - (1 - bar_width) / 2)

    ax2.set_ylabel("# Optimisation methods")
    ax2.set_xticks(x2)
    ax2.set_xticklabels(objective_counts.index, rotation=45, ha="right")

    label_bars(bars2, ax2)

    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
    ax2.spines["bottom"].set_bounds(-0.5 * x_step, n2)
    ax2.spines["left"].set_bounds(0, max(objective_counts.values))


with plt.rc_context(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial"],
        "axes.labelsize": 6,
        "axes.titlesize": 7,
        "xtick.labelsize": 5,
        "ytick.labelsize": 5,
        "lines.linewidth": 0.8,
        "patch.linewidth": 0.5,
    }
):
    fig = plt.figure(
        figsize=(18 / 2.54, 5),
        dpi=300,
    )  # layout="constrained")
    gs = fig.add_gridspec(
        3,
        2,
        width_ratios=[8, 2],
        height_ratios=[1, 0.5, 1],
        hspace=1.2,
        left=0.05,
        right=0.95,
        top=0.9,
    )

    ax_left = fig.add_subplot(gs[:, 0])  # left half, spans both rows
    ax_top = fig.add_subplot(gs[0, 1])  # right half top
    ax_bottom = fig.add_subplot(gs[2, 1])  # right half bottom

    optimizer_donuts(df_chart, ax=ax_left, fontsize_label=4, fontsize_leaf=5)
    subfigs_bc(optimizer_df, ax_top, ax_bottom)

    subfig_font = {
        "fontsize": 9,
        "fontweight": "bold",
    }
    fig.text(0.0, 1, "A", va="top", **subfig_font)
    fig.text(0.05, 1, "Optimisation methods", va="top", **subfig_font)
    fig.text(0.72, 1, "B", va="top", **subfig_font)
    fig.text(0.77, 1, "Optimiser\nimplementation", va="top", **subfig_font)
    fig.text(0.72, 0.48, "C", va="top", **subfig_font)
    fig.text(
        0.77,
        0.48,
        "Objective\nfunction\nimplementation",
        va="top",
        **subfig_font,
    )
    plt.savefig(OUTPUT_DIR / "Figure1.svg")
    plt.savefig(OUTPUT_DIR / "Figure1.pdf")